In [34]:
import requests
import pandas as pd
import numpy as np
import ast
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle


In [17]:
# 1. قراءة الملفين
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

# 2. إلقاء نظرة على أول 2 صفوف في كل ملف
print("Movies Shape:", movies.shape)
print("Credits Shape:", credits.shape)

movies.head(2)

Movies Shape: (4803, 20)
Credits Shape: (4803, 4)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500


In [18]:
# دمج الجدولين بناءً على اسم الفيلم أو الـ ID
movies = movies.merge(credits, on='title')

# التأكد من شكل الجدول الجديد والأعمدة المتاحة
print("Merged Shape:", movies.shape)
movies.info()

# احتفظ بالأعمدة المهمة فقط
# اختيار الأعمدة مع إضافة التقييم وعدد المصوتين
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew', 'vote_average', 'vote_count']]
movies.head()

Merged Shape: (4809, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,vote_count
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.2,11800
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",6.9,4500
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",6.3,4466
3,49026,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",7.6,9106
4,49529,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",6.1,2124


In [19]:
# فحص هل فيه قيم فاضية؟
print(movies.isnull().sum())

# التخلص من الصفوف اللي القصة فيها فاضية (عدد قليل جداً)
movies.dropna(inplace=True)

movie_id        0
title           0
overview        3
genres          0
keywords        0
cast            0
crew            0
vote_average    0
vote_count      0
dtype: int64


In [20]:
def convert(text):
    L = []
    # تحويل النص لـ List حقيقية
    for i in ast.literal_eval(text):
        L.append(i['name']) # بناخد بس القيمة اللي قدام المفتاح 'name'
    return L
# تطبيق الدالة على عمود genres
movies['genres'] = movies['genres'].apply(convert)

# تطبيق الدالة على عمود keywords
movies['keywords'] = movies['keywords'].apply(convert)

# نطبع أول 2 صفوف ونشوف الفرق
movies.head(2)

,movie_id,title,overview,genres,keywords,cast,crew,vote_average,vote_count
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",7.2,11800
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",6.9,4500


In [21]:
def convert_cast(text):
    L = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 3: # بناخد أول 3 ممثلين بس
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

# تطبيق الدالة على عمود cast
movies['cast'] = movies['cast'].apply(convert_cast)

In [22]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
            break # المخرج غالباً شخص واحد أو اثنين، فبنطلع أول ما نلاقيه
    return L

# تطبيق الدالة على عمود crew
movies['crew'] = movies['crew'].apply(fetch_director)

In [23]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [24]:
# طباعة أول صف لرؤية النتيجة بعد تنظيف الممثلين والمخرج والقصة
movies[['title', 'overview', 'genres', 'keywords', 'cast', 'crew']].head(1)

,title,overview,genres,keywords,cast,crew
0,Avatar,"[In, the, 22nd, century,, a, paraplegic, Marin...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",[James Cameron]


In [25]:
# دالة لإزالة المسافات بين الكلمات داخل كل قائمة
def collapse(L):
    L1 = []
    for i in L:
        L1.append(i.replace(" ", ""))
    return L1

# تطبيق الدالة على القوائم اللي فيها أسماء وكلمات مرادفة
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

In [26]:
# دمج القوائم في عمود واحد
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# إنشاء DataFrame جديد خفيف فيه الأعمدة اللي هتحتاجها للموديل والـ UI فقط
new_df = movies[['movie_id', 'title', 'tags', 'vote_average', 'vote_count']]

# تحويل قائمة الـ tags إلى نص واحد (String) مدموج بالمسافات
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

# تحويل كل الحروف لـ Lowercase عشان الموديل ما يفرقش بين Capital و Small
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

C:\Users\elomda\AppData\Local\Temp\ipykernel_16972\1153986234.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))
C:\Users\elomda\AppData\Local\Temp\ipykernel_16972\1153986234.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())


In [27]:
new_df['tags'][0]

'in the 22nd century, a paraplegic marine is dispatched to the moon pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. action adventure fantasy sciencefiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d samworthington zoesaldana sigourneyweaver jamescameron'

In [29]:
# 1. إنشاء الـ Vectorizer لاختيار أعلى 5000 كلمة شيوعاً مع إزالة كلمات الوصل (Stop Words زي is, the, in)
cv = CountVectorizer(max_features=5000, stop_words='english')

# 2. تحويل النص لـ NumPy Array
vectors = cv.fit_transform(new_df['tags']).toarray()

# شوف شكل الـ Vectors
print("Vectors Shape:", vectors.shape)

Vectors Shape: (4806, 5000)


In [31]:
# حساب التشابه بين كل فيلم وجميع الأفلام الأخرى
similarity = cosine_similarity(vectors)

# شوف شكل مصفوفة التشابه
print("Similarity Matrix Shape:", similarity.shape)

Similarity Matrix Shape: (4806, 4806)


In [32]:
def recommend(movie):
    # 1. إيجاد الـ Index الخاص بالفيلم
    movie_index = new_df[new_df['title'] == movie].index[0]
    
    # 2. جلب درجات التشابه للفيلم وترتيبها تنازلياً مع الاحتفاظ بالـ Index الأصلي
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    # 3. طباعة أسماء أعلى 5 أفلام مرشحة
    print(f"Top recommendations for '{movie}':\n")
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

In [33]:
# تجربة الدالة مع فيلم Avatar أو Inception أو The Dark Knight
recommend('Avatar')

Top recommendations for 'Avatar':

Titan A.E.
Small Soldiers
Independence Day
Ender's Game
Aliens vs Predator: Requiem


In [ ]:
# 1. حفظ جدول الأفلام بعد التنظيف كـ Dictionary
pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))

# 2. حفظ مصفوفة التشابه (Similarity Matrix)
pickle.dump(similarity, open('similarity.pkl', 'wb'))

: 